In [1]:
import ee
import geemap
from tgbs_rs.config.config import GEE_PROJECT

In [ ]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT) 

In [3]:
from tgbs_rs.config.config import AOI_PATHS
from tgbs_rs.utils import geojson_to_ee_geometry, load_site_feature
from tgbs_rs.data import build_baseline_layers
from tgbs_rs.data.baseline import summarize_baseline_layers_by_site
from tgbs_rs.config.config import PLOTTING_SCALE_DICT, DRIVE_FOLDER, DEFAULT_CRS
from tgbs_rs.io import (
    export_selected_layers_to_drive,
)
from tgbs_rs.utils import build_default_sites_featurecollection


In [22]:
Map = geemap.Map() 

In [9]:
kwale_aoi = geojson_to_ee_geometry(AOI_PATHS["kwale"])

In [8]:
layers = build_baseline_layers(kwale_aoi)

In [4]:
sites_fc = build_default_sites_featurecollection()

# Create Feature for biodiversity corridor
corridor_feature = load_site_feature(
    AOI_PATHS["bio_corridor"],
    site_id="bio_corridor",
    site_name="Biodiversity Corridor",
    site_category="corridor",
)

sites_fc = sites_fc.merge(ee.FeatureCollection([corridor_feature]))

all_sites_geom = sites_fc.geometry()

In [6]:
baseline_layers = build_baseline_layers(all_sites_geom)
baseline_layers.pop("forest_2000", None)
baseline_layers.pop("forest_loss", None)

In [7]:
baseline_summary_fc = summarize_baseline_layers_by_site(sites_fc)

In [ ]:
tasks = export_selected_layers_to_drive(
    layers=layers,
    aoi=kwale_aoi,
    layer_names=["dem", "slope", "hillshade", "land_cover", "soil_carbon", "bii_all", "canopy_height"],
    folder=DRIVE_FOLDER,
    scale_dict=PLOTTING_SCALE_DICT,
    crs=DEFAULT_CRS,
)

In [ ]:
task = export_selected_layers_to_drive(
    layers=layers,
    aoi=kwale_aoi,
    layer_names=["forest_2000", "forest_loss"],
    folder=DRIVE_FOLDER,
    scale_dict=PLOTTING_SCALE_DICT,
    crs=DEFAULT_CRS,
)

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=baseline_summary_fc,
    description="baseline_site_summaries",
    fileFormat="CSV",
    folder="DRIVE_FOLDER",
)
task.start()